### Step-1 Read bronze tables

In [0]:
student_info = spark.table("education.bronze_schema.student_info")
student_reg = spark.table("education.bronze_schema.student_registration")
assessments = spark.table("education.bronze_schema.assessments")
student_assessments = spark.table("education.bronze_schema.student_assessments")
courses = spark.table("education.bronze_schema.courses")
vle = spark.table("education.bronze_schema.vle")

### STEP 2: Remove Duplicates

In [0]:
silver_student_info = student_info.dropDuplicates()
silver_courses = courses.dropDuplicates()
silver_vle = vle.dropDuplicates()

### Imports

In [0]:
from pyspark.sql.functions import *

### student_info

In [0]:
silver_student_info = student_info.dropDuplicates() \
    .withColumn("final_result", trim(col("final_result"))) \
    .withColumn("age_band", trim(col("age_band"))) \
    .withColumn("imd_band", trim(col("imd_band"))) \
    .withColumn("gender", trim(col("gender"))) \
    .withColumn("region", trim(col("region"))) \
    .fillna({
        "num_of_prev_attempts": 0,
        "studied_credits": 0
    })

silver_student_info.write.format("delta").mode("overwrite") \
    .saveAsTable("education.silver_schema.student_info")

### student_registration

In [0]:
silver_student_registration = student_reg \
    .withColumn("date_registration", col("date_registration").cast("int")) \
    .withColumn("date_unregistration",
        when(col("date_unregistration").isNull(), 0)
        .otherwise(col("date_unregistration").cast("int"))
    )

silver_student_registration.write.format("delta").mode("overwrite") \
    .saveAsTable("education.silver_schema.student_registration")

### student_assessments

In [0]:
silver_student_assessments = student_assessments \
    .withColumn("score",
        when(col("score").isNull(), 0).otherwise(col("score"))
    )

silver_student_assessments.write.format("delta").mode("overwrite") \
    .saveAsTable("education.silver_schema.student_assessments")

### assessments

In [0]:
silver_assessments = assessments \
    .withColumn("weight", col("weight").cast("double"))

silver_assessments.write.format("delta").mode("overwrite") \
    .saveAsTable("education.silver_schema.assessments")

### courses

In [0]:
silver_courses = courses.dropDuplicates() \
    .withColumn("module_presentation_length",
                col("module_presentation_length").cast("int"))

silver_courses.write.format("delta").mode("overwrite") \
    .saveAsTable("education.silver_schema.courses")

### vle

In [0]:
silver_vle = vle.dropDuplicates()

silver_vle.write.format("delta").mode("overwrite") \
    .saveAsTable("education.silver_schema.vle")

### Join

In [0]:
s = silver_student_info.select(
    "id_student", "code_module", "code_presentation",
    "final_result", "num_of_prev_attempts"
)

r = silver_student_registration.select(
    "id_student", "code_module", "code_presentation",
    "date_registration", "date_unregistration"
)

sa = silver_student_assessments.select(
    "id_student", "id_assessment", "score"
)

a = silver_assessments.select(
    "id_assessment", "weight"
)

c = silver_courses.select(
    "code_module", "code_presentation", "module_presentation_length"
)

silver_student_full = s \
    .join(r, ["id_student", "code_module", "code_presentation"], "left") \
    .join(sa, "id_student", "left") \
    .join(a, "id_assessment", "left") \
    .join(c, ["code_module", "code_presentation"], "left")

### Enrichment

In [0]:
silver_student_enriched = silver_student_full \
    .withColumn(
        "enrollment_duration",
        coalesce(
            col("date_unregistration") - col("date_registration"),
            lit(0)
        )
    ) \
    .withColumn(
        "risk_flag",
        when((col("final_result") == "Fail") | (col("final_result") == "Withdrawn"), "High")
        .when(col("num_of_prev_attempts") > 0, "Medium")
        .otherwise("Low")
    ) \
    .withColumn(
        "weighted_score",
        (col("score") * col("weight")) / 100
    )

### Save Final Table

In [0]:
spark.sql("DROP TABLE IF EXISTS education.silver_schema.student_enriched")

silver_student_enriched.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("education.silver_schema.student_enriched")

### Total Number of Records in Silver Layer – Student Enriched Data

In [0]:
spark.sql("SELECT COUNT(*) FROM education.silver_schema.student_enriched").show()

+--------+
|COUNT(*)|
+--------+
|  213166|
+--------+

